<a href="https://colab.research.google.com/github/Shaheenovic/Drone-parking-monitoring-yolov8/blob/main/notebooks/02_training_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q ultralytics pyyaml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 5.1 MB/s eta 0:00:00


In [ ]:
!git clone https://github.com/Shaheenovic/Drone-parking-monitoring-yolov8.git
%cd Drone-parking-monitoring-yolov8
!git status

Cloning into 'Drone-parking-monitoring-yolov8'...
remote: Enumerating objects: 61, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 61 (delta 24), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (61/61), 34.79 KiB | 11.59 MiB/s, done.
Resolving deltas: 100% (24/24), done.
/content/Drone-parking-monitoring-yolov8
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [ ]:
from pathlib import Path
import urllib.request
import zipfile

OWNER = "Shaheenovic"
REPO = "Drone-parking-monitoring-yolov8"
TAG = "v1.0"
DATASET_ASSET = "drone-parking-v1-yolo11.zip"

url = f"https://github.com/{OWNER}/{REPO}/releases/download/{TAG}/{DATASET_ASSET}"

archive_path = Path("data") / DATASET_ASSET
extract_path = Path("data/raw")

archive_path.parent.mkdir(parents=True, exist_ok=True)
extract_path.mkdir(parents=True, exist_ok=True)

print("Downloading:", url)
urllib.request.urlretrieve(url, archive_path)

with zipfile.ZipFile(archive_path, "r") as z:
    z.extractall(extract_path)

print("Downloaded to:", archive_path)
print("Extracted to:", extract_path)

Downloading: https://github.com/Shaheenovic/Drone-parking-monitoring-yolov8/releases/download/v1.0/drone-parking-v1-yolo11.zip
Downloaded to: data/drone-parking-v1-yolo11.zip
Extracted to: data/raw


In [ ]:
from pathlib import Path
import hashlib
import shutil
import yaml

EXPECTED_SHA256 = "3F9E38E8AE7F7725F19F5F120165A1F19E8A54808D27690648A530CFB6B12999".lower()

sha256 = hashlib.sha256()

with open(archive_path, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        sha256.update(chunk)

actual_sha256 = sha256.hexdigest()

print("Expected SHA256:", EXPECTED_SHA256)
print("Actual SHA256:  ", actual_sha256)

if actual_sha256 != EXPECTED_SHA256:
    raise ValueError("SHA256 mismatch: أعد تنزيل ملف الـdataset ولا تبدأ التدريب.")

print("SHA256 verified successfully.")

yaml_files = list(extract_path.rglob("data.yaml"))

if not yaml_files:
    raise FileNotFoundError("لم يتم العثور على data.yaml بعد فك الضغط.")

print("\nFound data.yaml files:")
for p in yaml_files:
    print("-", p)

DATA_YAML = yaml_files[0].resolve()
DATASET_ROOT = DATA_YAML.parent.resolve()

print("\nDataset root:", DATASET_ROOT)
print("\nOriginal data.yaml:\n")
print(DATA_YAML.read_text())

Expected SHA256: 3f9e38e8ae7f7725f19f5f120165a1f19e8a54808d27690648a530cfb6b12999
Actual SHA256:   3f9e38e8ae7f7725f19f5f120165a1f19e8a54808d27690648a530cfb6b12999
SHA256 verified successfully.

Found data.yaml files:
- data/raw/data.yaml

Dataset root: /content/Drone-parking-monitoring-yolov8/data/raw

Original data.yaml:

train: ../train/images
val: ../valid/images
test: ../test/images

nc: 4
names: ['Empty', 'Illegal', 'LicensePlate', 'Occupied']

roboflow:
  workspace: eng-ahmed_shaheen-hotmail-com
  project: drone-parking-monitoring-yolov8
  version: 1
  license: CC BY 4.0
  url: https://universe.roboflow.com/eng-ahmed_shaheen-hotmail-com/drone-parking-monitoring-yolov8/dataset/1


In [ ]:
with open(DATA_YAML, "r") as f:
    data_config = yaml.safe_load(f)

data_config["path"] = str(DATASET_ROOT)

for split in ["train", "val", "test"]:
    if split in data_config:
        print(f"{split}: {data_config[split]}")

FIXED_DATA_YAML = Path("data") / "data_colab.yaml"

with open(FIXED_DATA_YAML, "w") as f:
    yaml.safe_dump(data_config, f, sort_keys=False)

print("\nSaved corrected config to:", FIXED_DATA_YAML)
print("\nCorrected data.yaml:\n")
print(FIXED_DATA_YAML.read_text())

train: ../train/images
val: ../valid/images
test: ../test/images

Saved corrected config to: data/data_colab.yaml

Corrected data.yaml:

train: ../train/images
val: ../valid/images
test: ../test/images
nc: 4
names:
- Empty
- Illegal
- LicensePlate
- Occupied
roboflow:
  workspace: eng-ahmed_shaheen-hotmail-com
  project: drone-parking-monitoring-yolov8
  version: 1
  license: CC BY 4.0
  url: https://universe.roboflow.com/eng-ahmed_shaheen-hotmail-com/drone-parking-monitoring-yolov8/dataset/1
path: /content/Drone-parking-monitoring-yolov8/data/raw



In [ ]:
from pathlib import Path

for split in ["train", "valid", "val", "test"]:
    image_dir = DATASET_ROOT / split / "images"
    label_dir = DATASET_ROOT / split / "labels"

    if image_dir.exists():
        images = list(image_dir.glob("*.*"))
        labels = list(label_dir.glob("*.txt")) if label_dir.exists() else []

        print(f"{split}:")
        print(f"  Images: {len(images)}")
        print(f"  Labels: {len(labels)}")

train:
  Images: 401
  Labels: 401
valid:
  Images: 94
  Labels: 94


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=str(FIXED_DATA_YAML),
    epochs=30,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    project="runs",
    name="drone_parking_yolov8n",
    exist_ok=True,
    pretrained=True,
    optimizer="auto",
    seed=42,
    deterministic=True,
    patience=10,
    verbose=True
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.163 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/data_colab.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=No

In [10]:
from pathlib import Path
from ultralytics import YOLO

run_dir = Path("runs/detect/runs/drone_parking_yolov8n")
best_model_path = run_dir / "weights" / "best.pt"

print("Run directory:", run_dir.resolve())
print("Best model exists:", best_model_path.exists())

if not best_model_path.exists():
    raise FileNotFoundError(f"best.pt was not found at: {best_model_path.resolve()}")

best_model = YOLO(str(best_model_path))

metrics = best_model.val(
    data=str(FIXED_DATA_YAML),
    split="val"
)

print("\nFinal validation metrics")
print(f"Precision:  {metrics.box.mp:.3f}")
print(f"Recall:     {metrics.box.mr:.3f}")
print(f"mAP50:      {metrics.box.map50:.3f}")
print(f"mAP50-95:   {metrics.box.map:.3f}")

Run directory: /content/Drone-parking-monitoring-yolov8/runs/detect/runs/drone_parking_yolov8n
Best model exists: True
Ultralytics 8.4.163 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1598.7±318.9 MB/s, size: 67.9 KB)
val: Scanning /content/Drone-parking-monitoring-yolov8/data/raw/valid/labels.cache... 94 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 94/94 24.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.9it/s 3.1s
                   all         94        385      0.917      0.888      0.903       0.72
                 Empty          6          6      0.799      0.667      0.649      0.588
               Illegal         94        110      0.982      0.978      0.992      0.886
          LicensePlate         78        128      0.911      0.953       0.98  

In [11]:
from pathlib import Path
import shutil

source_dir = Path("runs/detect/runs/drone_parking_yolov8n")
output_dir = Path("/content/training_results")

output_dir.mkdir(parents=True, exist_ok=True)

files_to_copy = [
    "results.csv",
    "results.png",
    "PR_curve.png",
    "F1_curve.png",
    "P_curve.png",
    "R_curve.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "labels.jpg",
]

for filename in files_to_copy:
    source_file = source_dir / filename

    if source_file.exists():
        shutil.copy2(source_file, output_dir / filename)
        print(f"Copied: {filename}")
    else:
        print(f"Not found: {filename}")

print("\nFiles ready for GitHub upload:")
for file_path in sorted(output_dir.iterdir()):
    print("-", file_path.name)

Copied: results.csv
Copied: results.png
Not found: PR_curve.png
Not found: F1_curve.png
Not found: P_curve.png
Not found: R_curve.png
Copied: confusion_matrix.png
Copied: confusion_matrix_normalized.png
Copied: labels.jpg

Files ready for GitHub upload:
- confusion_matrix.png
- confusion_matrix_normalized.png
- labels.jpg
- results.csv
- results.png


In [12]:
from pathlib import Path
from ultralytics import YOLO
import shutil

best_model = YOLO("runs/detect/runs/drone_parking_yolov8n/weights/best.pt")

val_images_dir = DATASET_ROOT / "valid" / "images"
val_images = sorted(
    path for path in val_images_dir.iterdir()
    if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
)

selected_images = val_images[:10]

validation_output = Path("/content/validation_predictions")
validation_output.mkdir(parents=True, exist_ok=True)

for image_path in selected_images:
    prediction = best_model.predict(
        source=str(image_path),
        conf=0.25,
        save=True,
        project="/content/prediction_runs",
        name="validation",
        exist_ok=True,
        verbose=False
    )

prediction_dir = Path("/content/prediction_runs/validation")

for prediction_image in prediction_dir.iterdir():
    if prediction_image.suffix.lower() in {".jpg", ".jpeg", ".png"}:
        shutil.copy2(prediction_image, validation_output / prediction_image.name)

print(f"Selected validation images: {len(selected_images)}")
print(f"Saved prediction images: {len(list(validation_output.iterdir()))}")
print("Folder:", validation_output)

Results saved to /content/prediction_runs/validation
Results saved to /content/prediction_runs/validation
Results saved to /content/prediction_runs/validation
Results saved to /content/prediction_runs/validation
Results saved to /content/prediction_runs/validation
Results saved to /content/prediction_runs/validation
Results saved to /content/prediction_runs/validation
Results saved to /content/prediction_runs/validation
Results saved to /content/prediction_runs/validation
Results saved to /content/prediction_runs/validation
Selected validation images: 10
Saved prediction images: 10
Folder: /content/validation_predictions


In [13]:
from pathlib import Path
from PIL import Image, ImageDraw
import shutil

annotation_output = Path("/content/annotation_examples")
annotation_output.mkdir(parents=True, exist_ok=True)

class_names = ["Empty", "Illegal", "LicensePlate", "Occupied"]
colors = ["red", "orange", "blue", "lime"]

selected_annotation_images = val_images[:5]

for image_path in selected_annotation_images:
    label_path = (
        DATASET_ROOT
        / "valid"
        / "labels"
        / f"{image_path.stem}.txt"
    )

    image = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(image)

    width, height = image.size

    if label_path.exists():
        for line in label_path.read_text().strip().splitlines():
            class_id, x_center, y_center, box_width, box_height = map(float, line.split())

            class_id = int(class_id)

            x1 = (x_center - box_width / 2) * width
            y1 = (y_center - box_height / 2) * height
            x2 = (x_center + box_width / 2) * width
            y2 = (y_center + box_height / 2) * height

            color = colors[class_id]
            label = class_names[class_id]

            draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
            draw.text((x1, max(0, y1 - 15)), label, fill=color)

    output_path = annotation_output / image_path.name
    image.save(output_path)

print(f"Annotation examples saved: {len(list(annotation_output.iterdir()))}")
print("Folder:", annotation_output)

Annotation examples saved: 5
Folder: /content/annotation_examples


In [14]:
from pathlib import Path

new_images_dir = Path("/content/new_images")
new_images_dir.mkdir(parents=True, exist_ok=True)

print("Folder ready:", new_images_dir)

Folder ready: /content/new_images


In [15]:
from pathlib import Path
from ultralytics import YOLO
import shutil

best_model = YOLO("runs/detect/runs/drone_parking_yolov8n/weights/best.pt")

new_images_dir = Path("/content/new_images")
new_image_files = sorted(
    path for path in new_images_dir.iterdir()
    if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
)

if len(new_image_files) != 5:
    raise ValueError(f"Expected 5 images, but found {len(new_image_files)}.")

new_predictions_output = Path("/content/new_image_predictions")
new_predictions_output.mkdir(parents=True, exist_ok=True)

for image_path in new_image_files:
    best_model.predict(
        source=str(image_path),
        conf=0.25,
        save=True,
        project="/content/prediction_runs",
        name="new_images",
        exist_ok=True,
        verbose=False
    )

prediction_dir = Path("/content/prediction_runs/new_images")

for prediction_image in prediction_dir.iterdir():
    if prediction_image.suffix.lower() in {".jpg", ".jpeg", ".png"}:
        shutil.copy2(
            prediction_image,
            new_predictions_output / prediction_image.name
        )

print(f"New source images: {len(new_image_files)}")
print(f"New-image predictions saved: {len(list(new_predictions_output.iterdir()))}")
print("Folder:", new_predictions_output)

Results saved to /content/prediction_runs/new_images
Results saved to /content/prediction_runs/new_images
Results saved to /content/prediction_runs/new_images
Results saved to /content/prediction_runs/new_images
Results saved to /content/prediction_runs/new_images
New source images: 5
New-image predictions saved: 5
Folder: /content/new_image_predictions
